# STIR-Net V1 — 15 hierarchical temporal-memory real overfit

This notebook tests the newly implemented hierarchical temporal-memory architecture on the same **full all-cell BlastoSPIM first-overfit scene** used by the earlier STIR-Net debugging notebooks.

It is deliberately a **causal diagnostic**, not another generic "does the loss go down?" experiment.

The central question is:

> Does the complete five-frame temporal memory become **different temporal identities for the split hypotheses of a real merged source**, and if not, where does that identity disappear?

The notebook:

- reuses the Notebook-12 `spatial_dense` checkpoint;
- migrates it strictly into the current hierarchical-memory model;
- starts directly at `temporal_dense`;
- trains the existing five-stage curriculum from step 30 to 85;
- takes snapshots at steps 30 / 50 / 70 / 80 / 85;
- traces component→memory and query→memory attention for **source 9**;
- measures attention diversity, embedding diversity, center separation, mask similarity, and Hungarian assignment churn;
- performs same-weight causal ablations:
  - `full`
  - `zero_node`
  - `shuffle_node`
  - `tracklet_only`
  - `node_only`
  - `accepted_only`
  - `legacy_like_graph`
- compares the learned detection-node representation under the full, accepted-only, and legacy-like graphs.

No architecture is modified in this notebook.

In [ ]:
from pathlib import Path
from collections import defaultdict
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.debugging.probes.matching import run_matching_probe
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
    QUERY_TEMPORAL,
    QUERY_DISCOVERY,
)
from learned.stirnet.training.checkpoint import (
    load_checkpoint,
    save_checkpoint,
)
from learned.stirnet.training.trainer import (
    Trainer,
    model_forward_from_batch,
    move_to_device,
)

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16
LOG_EVERY = 5
ASSIGNMENT_EVERY = 1

QUERY_NAMES = {
    QUERY_PRIMARY: "primary",
    QUERY_SPLIT: "split",
    QUERY_TEMPORAL: "temporal",
    QUERY_DISCOVERY: "discovery",
}

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "15_hierarchical_temporal_memory"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

SPATIAL_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 15 requires CUDA.")

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Checkpoint :", SPATIAL_CHECKPOINT)

## 1. Load the current v3 real scene and validate the temporal contract

The current cache should contain every detection from the five-frame window and a complete within-sample directed candidate graph.

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
target = batch["targets"][0]

cfg = _reduced_config()
cfg.curriculum.enabled = True
cfg.curriculum.spatial_dense_steps = 30
cfg.curriculum.temporal_dense_steps = 20
cfg.curriculum.query_bootstrap_steps = 20
cfg.curriculum.native_bootstrap_steps = 10

assert sample["current_count"] == 36, sample
assert sample["target_count"] == 33, sample
assert sample["temporal_tracklets"] == 52, sample

N = int(batch["graph_x"].shape[0])
E = int(batch["graph_edge_index"].shape[1])
M = int(batch["temporal_ref_um"].shape[0])
accepted_edges = int((batch["graph_edge_attr"][:, 14] > 0.5).sum())

node_times = batch["node_time_offset"].detach().cpu().numpy().round().astype(int)
frame_counts = {
    int(t): int((node_times == t).sum())
    for t in sorted(np.unique(node_times))
}

print("Sample:", sample)
print()
print("ROI shape                :", sample["roi_shape"])
print("Current components       :", sample["current_count"])
print("GT cells                 :", sample["target_count"])
print("Detection nodes          :", N)
print("Node counts / frame      :", frame_counts)
print("Tracklets                :", M)
print("Candidate edges          :", E)
print("Accepted-direction edges :", accepted_edges)
print("Required queries         :", sample["required_queries"])
print("Source 9 companions      :", sample["split_companions_by_source"].get(SOURCE_ID))
print("History valid            :", int(batch["node_history_valid"].sum()), "/", N)

assert E == N * (N - 1), (
    f"Expected the complete directed candidate graph {N*(N-1)}, got {E}."
)
assert set(frame_counts) == {-2, -1, 0, 1, 2}
assert sample["split_companions_by_source"].get(SOURCE_ID) == 8
assert batch["graph_edge_attr"].shape[1] == 15

node_table = pd.DataFrame({
    "node_index": np.arange(N),
    "node_id": batch["node_ids"].detach().cpu().numpy(),
    "time": node_times,
    "tracklet_id": batch["tracklet_id"].detach().cpu().numpy(),
    "history_valid": batch["node_history_valid"].detach().cpu().numpy(),
    "observed_z_um": batch["node_observed_ref_um"][:, 0].detach().cpu().numpy(),
    "observed_y_um": batch["node_observed_ref_um"][:, 1].detach().cpu().numpy(),
    "observed_x_um": batch["node_observed_ref_um"][:, 2].detach().cpu().numpy(),
})
display(node_table.head(20))

## 2. Prepare GPU batch and migrate the Notebook-12 spatial checkpoint

We intentionally load **model state only**. The new temporal-memory modules did not exist in the old optimizer, so this experiment starts a fresh optimizer while preserving the learned spatial weights.

The first actual optimizer step is curriculum step 30, i.e. `temporal_dense`.

In [ ]:
if not SPATIAL_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Notebook-12 spatial checkpoint was not found:\n"
        f"{SPATIAL_CHECKPOINT}\n\n"
        "Run Notebook 12 first or update SPATIAL_CHECKPOINT to the correct file."
    )

def prepare_device_batch(cpu_batch):
    out = {}
    for key, value in cpu_batch.items():
        if key == "targets":
            out[key] = value
        elif key == "spatial_inputs":
            out[key] = value.to(
                device=device,
                dtype=AMP_DTYPE,
                non_blocking=True,
            )
        elif key == "instance_labels":
            out[key] = value.to(
                device=device,
                dtype=torch.int32,
                non_blocking=True,
            )
        else:
            out[key] = move_to_device(value, device)
    return out

device_batch = prepare_device_batch(batch)

model = StirNet(cfg).to(device)

loaded = load_checkpoint(
    SPATIAL_CHECKPOINT,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

migration_notes = loaded.get("temporal_migration", loaded.get("history_migration", []))
print("Loaded checkpoint step:", loaded.get("step"))
print("Migration notes:", len(migration_notes))
for note in migration_notes[:20]:
    print(" ", note)
if len(migration_notes) > 20:
    print(" ...", len(migration_notes) - 20, "more")

trainer = Trainer(
    model,
    cfg,
    device=device,
    amp_dtype="fp16",
)
trainer.global_step = 30

# Independent full-objective evaluator. The Trainer criterion changes its
# active weights with the curriculum; this one does not.
eval_criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(device).eval()

print()
print("Trainer global step:", trainer.global_step)
print("Next curriculum stage should be temporal_dense.")

## 3. Diagnostic helpers

The helpers below deliberately measure several representations separately:

1. temporal attention diversity;
2. query embedding diversity;
3. center separation;
4. coarse-mask sibling similarity;
5. native sibling similarity;
6. structured Hungarian assignment.

This allows us to localize where identity collapses.

In [ ]:
def hard_dice(pred, gt, eps=1e-6):
    pred = pred.bool()
    gt = gt.bool()
    inter = (pred & gt).sum().float()
    return float((2 * inter + eps) / (pred.sum() + gt.sum() + eps))

def binary_auc(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    pos = labels == 1
    neg = labels == 0
    n_pos = int(pos.sum())
    n_neg = int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = rankdata(scores)
    return float(
        (ranks[pos].sum() - n_pos * (n_pos + 1) / 2)
        / (n_pos * n_neg)
    )

def offdiag_values(matrix):
    matrix = torch.as_tensor(matrix)
    if matrix.shape[0] < 2:
        return matrix.new_zeros((0,))
    mask = ~torch.eye(
        matrix.shape[0],
        dtype=torch.bool,
        device=matrix.device,
    )
    return matrix[mask]

def pairwise_cosine_summary(tokens):
    x = torch.as_tensor(tokens).float()
    if x.shape[0] < 2:
        return {
            "mean": float("nan"),
            "median": float("nan"),
            "min": float("nan"),
            "max": float("nan"),
        }
    x = F.normalize(x, dim=-1, eps=1e-8)
    values = offdiag_values(x @ x.T)
    return {
        "mean": float(values.mean()),
        "median": float(values.median()),
        "min": float(values.min()),
        "max": float(values.max()),
    }

def pairwise_soft_dice_summary(probabilities):
    p = torch.as_tensor(probabilities).float().flatten(1)
    if p.shape[0] < 2:
        return {
            "mean": float("nan"),
            "median": float("nan"),
            "min": float("nan"),
            "max": float("nan"),
        }
    inter = p @ p.T
    sums = p.sum(dim=1)
    dice = (2 * inter + 1e-6) / (
        sums[:, None] + sums[None, :] + 1e-6
    )
    values = offdiag_values(dice)
    return {
        "mean": float(values.mean()),
        "median": float(values.median()),
        "min": float(values.min()),
        "max": float(values.max()),
    }

def js_divergence_pairwise(distributions):
    p = torch.as_tensor(distributions).float()
    if p.shape[0] < 2:
        return float("nan")
    p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-12)
    values = []
    for i in range(p.shape[0]):
        for j in range(i + 1, p.shape[0]):
            m = 0.5 * (p[i] + p[j])
            kl_i = (
                p[i]
                * (
                    p[i].clamp_min(1e-12).log()
                    - m.clamp_min(1e-12).log()
                )
            ).sum()
            kl_j = (
                p[j]
                * (
                    p[j].clamp_min(1e-12).log()
                    - m.clamp_min(1e-12).log()
                )
            ).sum()
            values.append(0.5 * (kl_i + kl_j))
    return float(torch.stack(values).mean())

def source_query_labels(output, source_id=SOURCE_ID):
    q_types = output.query_types[0].detach().cpu()
    q_sources = output.source_instance_ids[0].detach().cpu()
    seeded = (
        (q_sources == source_id)
        & (
            (q_types == QUERY_PRIMARY)
            | (q_types == QUERY_SPLIT)
        )
    )
    indices = torch.nonzero(
        seeded,
        as_tuple=False,
    ).flatten().tolist()

    primary = [
        q for q in indices
        if int(q_types[q]) == QUERY_PRIMARY
    ]
    splits = [
        q for q in indices
        if int(q_types[q]) == QUERY_SPLIT
    ]

    labels = {}
    for q in primary:
        labels[int(q)] = "primary"
    for slot, q in enumerate(sorted(splits)):
        labels[int(q)] = f"split{slot}"

    return indices, labels

def source_assignment_rows(output, probe, step, stage, source_id=SOURCE_ID):
    q_indices, labels = source_query_labels(output, source_id)
    gt_ids = target["ids"].detach().cpu()
    rows = []
    mapping = probe.query_to_target

    for q in q_indices:
        target_index = mapping.get((0, int(q)))
        rows.append({
            "step": int(step),
            "stage": str(stage),
            "query": int(q),
            "query_label": labels[int(q)],
            "query_type": QUERY_NAMES[int(output.query_types[0, q].detach().cpu())],
            "gt_id": (
                -1
                if target_index is None
                else int(gt_ids[target_index])
            ),
            "exist_prob": float(
                output.exist_logits[0, q].sigmoid().detach().cpu()
            ),
            "center_z": float(
                output.centers_cellscale[0, q, 0].detach().cpu()
            ),
            "center_y": float(
                output.centers_cellscale[0, q, 1].detach().cpu()
            ),
            "center_x": float(
                output.centers_cellscale[0, q, 2].detach().cpu()
            ),
        })
    return rows

In [ ]:
def attention_diversity(distributions, node_tracklets=None):
    p = torch.as_tensor(distributions).float()
    if p.numel() == 0 or p.shape[0] < 2:
        return {
            "attention_cos_mean": float("nan"),
            "attention_cos_median": float("nan"),
            "attention_js_mean": float("nan"),
            "unique_top_nodes": 0,
            "unique_top_tracklets": 0,
            "normalized_entropy_mean": float("nan"),
        }

    normalized = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-12)
    cosine = F.normalize(normalized, dim=-1) @ F.normalize(
        normalized,
        dim=-1,
    ).T
    cos_values = offdiag_values(cosine)
    entropy = -(
        normalized
        * normalized.clamp_min(1e-12).log()
    ).sum(dim=-1)
    entropy = entropy / max(
        math.log(max(normalized.shape[1], 2)),
        1e-8,
    )
    top_nodes = normalized.argmax(dim=-1)

    if node_tracklets is None:
        unique_tracklets = 0
    else:
        node_tracklets = torch.as_tensor(node_tracklets).long()
        unique_tracklets = int(
            torch.unique(node_tracklets[top_nodes]).numel()
        )

    return {
        "attention_cos_mean": float(cos_values.mean()),
        "attention_cos_median": float(cos_values.median()),
        "attention_js_mean": js_divergence_pairwise(normalized),
        "unique_top_nodes": int(torch.unique(top_nodes).numel()),
        "unique_top_tracklets": unique_tracklets,
        "normalized_entropy_mean": float(entropy.mean()),
    }

def component_attention_table(output, source_id=SOURCE_ID):
    debug = output.debug or {}
    component_debug = debug.get("component_temporal_attention")
    node_memory = debug.get("node_memory")

    if (
        not component_debug
        or not node_memory
        or "node" not in component_debug
        or "full_weights" not in component_debug["node"]
    ):
        return pd.DataFrame()

    instance_ids = (
        component_debug["component_instance_ids"]
        .detach()
        .cpu()
        .long()
    )
    rows = torch.nonzero(
        instance_ids == source_id,
        as_tuple=False,
    ).flatten()
    if len(rows) != 1:
        return pd.DataFrame()

    row = int(rows[0])
    weights = (
        component_debug["node"]["full_weights"][row]
        .float()
        .detach()
        .cpu()
    )

    node_ids = node_memory["node_ids"]
    node_ids = (
        torch.arange(len(weights))
        if node_ids is None
        else node_ids.detach().cpu()
    )
    times = node_memory["time_offset"].detach().cpu()
    tracklets = node_memory["tracklet_id"].detach().cpu()
    observed = node_memory["observed_ref_um"].detach().cpu()
    projected = node_memory["projected_ref_um"].detach().cpu()

    order = torch.argsort(weights, descending=True)
    records = []
    for rank, node_index in enumerate(order[:15].tolist(), start=1):
        records.append({
            "rank": rank,
            "node_index": int(node_index),
            "node_id": int(node_ids[node_index]),
            "time": int(round(float(times[node_index]))),
            "tracklet_id": int(tracklets[node_index]),
            "weight": float(weights[node_index]),
            "observed_z_um": float(observed[node_index, 0]),
            "observed_y_um": float(observed[node_index, 1]),
            "observed_x_um": float(observed[node_index, 2]),
            "projected_z_um": float(projected[node_index, 0]),
            "projected_y_um": float(projected[node_index, 1]),
            "projected_x_um": float(projected[node_index, 2]),
        })

    frame_mass = {
        f"mass_t{int(t):+d}": float(
            weights[times.round().long() == int(t)].sum()
        )
        for t in (-2, -1, 0, 1, 2)
    }
    table = pd.DataFrame(records)
    for key, value in frame_mass.items():
        table.attrs[key] = value
    table.attrs["entropy"] = float(
        component_debug["node"]["entropy"][row].detach().cpu()
    )
    table.attrs["max_weight"] = float(
        component_debug["node"]["max_weight"][row].detach().cpu()
    )
    return table

def query_attention_analysis(output, source_id=SOURCE_ID):
    debug = output.debug or {}
    node_memory = debug.get("node_memory")
    layers = debug.get("query_temporal_attention") or []

    if not node_memory:
        return pd.DataFrame(), pd.DataFrame(), {}

    node_ids = node_memory["node_ids"]
    node_ids = (
        torch.arange(len(node_memory["time_offset"]))
        if node_ids is None
        else node_ids.detach().cpu()
    )
    times = node_memory["time_offset"].detach().cpu()
    tracklets = node_memory["tracklet_id"].detach().cpu()
    observed = node_memory["observed_ref_um"].detach().cpu()
    projected = node_memory["projected_ref_um"].detach().cpu()

    _, qlabels = source_query_labels(output, source_id)
    top_rows = []
    frame_rows = []
    diversity = {}

    for layer_index, layer_debug in enumerate(layers):
        if (
            not layer_debug
            or "node" not in layer_debug
            or "full_weights" not in layer_debug["node"]
        ):
            continue

        slots = layer_debug["query_slot_index"].detach().cpu().long()
        qtypes = layer_debug["query_type"].detach().cpu().long()
        sources = layer_debug["source_instance_id"].detach().cpu().long()
        selected_rows = torch.nonzero(
            (sources == source_id)
            & (
                (qtypes == QUERY_PRIMARY)
                | (qtypes == QUERY_SPLIT)
            ),
            as_tuple=False,
        ).flatten()

        if selected_rows.numel() == 0:
            continue

        full = (
            layer_debug["node"]["full_weights"][selected_rows]
            .float()
            .detach()
            .cpu()
        )
        selected_slots = slots[selected_rows]

        tracklet_full = None
        if (
            "tracklet" in layer_debug
            and "full_weights" in layer_debug["tracklet"]
        ):
            tracklet_full = (
                layer_debug["tracklet"]["full_weights"][selected_rows]
                .float()
                .detach()
                .cpu()
            )

        diversity[layer_index] = attention_diversity(
            full,
            tracklets,
        )
        diversity[layer_index]["historical_attention_mass_mean"] = float(
            full[:, times.round().long() != 0].sum(dim=-1).mean()
        )

        for local_row, qslot in enumerate(selected_slots.tolist()):
            weights = full[local_row]
            top_index = int(weights.argmax())
            top_tracklet = (
                -1
                if tracklet_full is None or tracklet_full.shape[1] == 0
                else int(tracklet_full[local_row].argmax())
            )
            top_tracklet_weight = (
                float("nan")
                if tracklet_full is None or tracklet_full.shape[1] == 0
                else float(tracklet_full[local_row, top_tracklet])
            )

            top_rows.append({
                "layer": layer_index,
                "query": int(qslot),
                "query_label": qlabels.get(int(qslot), f"q{qslot}"),
                "query_type": QUERY_NAMES[int(qtypes[selected_rows[local_row]])],
                "top_node_index": top_index,
                "top_node_id": int(node_ids[top_index]),
                "top_time": int(round(float(times[top_index]))),
                "top_node_tracklet": int(tracklets[top_index]),
                "top_node_weight": float(weights[top_index]),
                "top_tracklet": top_tracklet,
                "top_tracklet_weight": top_tracklet_weight,
                "entropy": float(
                    layer_debug["node"]["entropy"][selected_rows[local_row]]
                    .detach()
                    .cpu()
                ),
                "historical_mass": float(
                    weights[times.round().long() != 0].sum()
                ),
                "top_observed_z_um": float(observed[top_index, 0]),
                "top_observed_y_um": float(observed[top_index, 1]),
                "top_observed_x_um": float(observed[top_index, 2]),
                "top_projected_z_um": float(projected[top_index, 0]),
                "top_projected_y_um": float(projected[top_index, 1]),
                "top_projected_x_um": float(projected[top_index, 2]),
            })

            frame_record = {
                "layer": layer_index,
                "query": int(qslot),
                "query_label": qlabels.get(int(qslot), f"q{qslot}"),
            }
            rounded_time = times.round().long()
            for t in (-2, -1, 0, 1, 2):
                frame_record[f"t{t:+d}"] = float(
                    weights[rounded_time == t].sum()
                )
            frame_rows.append(frame_record)

    return (
        pd.DataFrame(top_rows),
        pd.DataFrame(frame_rows),
        diversity,
    )

In [ ]:
def source_representation_metrics(output, source_id=SOURCE_ID):
    q_indices, _ = source_query_labels(output, source_id)
    idx = torch.as_tensor(
        q_indices,
        device=output.query_embeddings.device,
        dtype=torch.long,
    )

    embeddings = [
        aux["query_embeddings"][0, idx].detach().float().cpu()
        for aux in output.aux_outputs
    ] + [
        output.query_embeddings[0, idx].detach().float().cpu()
    ]

    centers = (
        output.debug["query_layer_references_cellscale"][:, 0, idx]
        .detach()
        .float()
        .cpu()
        * float(output.dref_um[0].detach().cpu())
    )

    coarse_logits = [
        aux["coarse_mask_logits"][0, idx].detach().float().cpu()
        for aux in output.aux_outputs
    ] + [
        output.coarse_mask_logits[0, idx].detach().float().cpu()
    ]

    rows = []
    for layer in range(len(embeddings)):
        emb_stats = pairwise_cosine_summary(embeddings[layer])

        distances = (
            torch.pdist(centers[layer])
            if centers[layer].shape[0] > 1
            else torch.zeros(0)
        )
        if distances.numel():
            center_stats = {
                "center_mean_um": float(distances.mean()),
                "center_median_um": float(distances.median()),
                "center_min_um": float(distances.min()),
                "center_max_um": float(distances.max()),
            }
        else:
            center_stats = {
                "center_mean_um": float("nan"),
                "center_median_um": float("nan"),
                "center_min_um": float("nan"),
                "center_max_um": float("nan"),
            }

        probs = coarse_logits[layer].sigmoid()
        mask_stats = pairwise_soft_dice_summary(probs)

        rows.append({
            "layer": layer,
            "embedding_cos_mean": emb_stats["mean"],
            "embedding_cos_median": emb_stats["median"],
            "coarse_pair_dice_mean": mask_stats["mean"],
            "coarse_pair_dice_median": mask_stats["median"],
            **center_stats,
        })

    return pd.DataFrame(rows)

@torch.no_grad()
def source_native_metrics(model, output, probe, source_id=SOURCE_ID, chunk_voxels=524_288):
    q_indices, qlabels = source_query_labels(output, source_id)
    idx = torch.as_tensor(
        q_indices,
        device=output.query_embeddings.device,
        dtype=torch.long,
    )

    if idx.numel() == 0:
        return {
            "native_pair_dice_mean": float("nan"),
            "native_pair_dice_median": float("nan"),
            "native_pair_dice_max": float("nan"),
            "assigned_mean_dice": float("nan"),
            "matched_seeded": 0,
        }, pd.DataFrame()

    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        native_logits = model.render_masks(
            output,
            [idx],
        )[0].flatten(1)

    q_count, voxel_count = native_logits.shape
    pred_sum = torch.zeros(q_count, device=device, dtype=torch.float64)
    pair_inter = torch.zeros(
        q_count,
        q_count,
        device=device,
        dtype=torch.float64,
    )

    mapping = probe.query_to_target
    gt_ids = target["ids"].detach().cpu()
    gt_map = target["label_map"].to(
        device=device,
        dtype=torch.int32,
        non_blocking=True,
    ).flatten()

    assigned_pred_sum = torch.zeros(q_count, device=device, dtype=torch.float64)
    assigned_gt_sum = torch.zeros(q_count, device=device, dtype=torch.float64)
    assigned_inter = torch.zeros(q_count, device=device, dtype=torch.float64)
    assigned_target = [-1] * q_count

    for local_index, q in enumerate(q_indices):
        target_index = mapping.get((0, int(q)))
        if target_index is not None:
            assigned_target[local_index] = int(target_index)

    for start in range(0, voxel_count, chunk_voxels):
        stop = min(start + chunk_voxels, voxel_count)
        probs = native_logits[:, start:stop].float().sigmoid()

        pred_sum += probs.sum(dim=1).double()
        pair_inter += (probs @ probs.T).double()

        gt_chunk = gt_map[start:stop]
        for local_index, target_index in enumerate(assigned_target):
            if target_index < 0:
                continue
            gt_id = int(gt_ids[target_index])
            gt = (gt_chunk == gt_id).float()
            p = probs[local_index]
            assigned_pred_sum[local_index] += p.sum().double()
            assigned_gt_sum[local_index] += gt.sum().double()
            assigned_inter[local_index] += (p * gt).sum().double()

        del probs, gt_chunk

    pair_dice = (
        2 * pair_inter + 1e-6
    ) / (
        pred_sum[:, None] + pred_sum[None, :] + 1e-6
    )
    pair_values = offdiag_values(pair_dice.float())

    query_rows = []
    assigned_values = []
    for local_index, q in enumerate(q_indices):
        target_index = assigned_target[local_index]
        if target_index < 0:
            dice = float("nan")
            gt_id = -1
        else:
            dice = float(
                (
                    2 * assigned_inter[local_index] + 1e-6
                )
                / (
                    assigned_pred_sum[local_index]
                    + assigned_gt_sum[local_index]
                    + 1e-6
                )
            )
            gt_id = int(gt_ids[target_index])
            assigned_values.append(dice)

        query_rows.append({
            "query": int(q),
            "query_label": qlabels[int(q)],
            "assigned_gt_id": gt_id,
            "assigned_native_dice": dice,
        })

    result = {
        "native_pair_dice_mean": (
            float(pair_values.mean())
            if pair_values.numel()
            else float("nan")
        ),
        "native_pair_dice_median": (
            float(pair_values.median())
            if pair_values.numel()
            else float("nan")
        ),
        "native_pair_dice_max": (
            float(pair_values.max())
            if pair_values.numel()
            else float("nan")
        ),
        "assigned_mean_dice": (
            float(np.mean(assigned_values))
            if assigned_values
            else float("nan")
        ),
        "matched_seeded": int(len(assigned_values)),
    }

    del native_logits, gt_map
    torch.cuda.empty_cache()

    return result, pd.DataFrame(query_rows)

In [ ]:
@torch.no_grad()
def run_full_forward(
    model,
    b,
    *,
    memory_ablation="full",
    graph_ablation="full",
    full_attention=True,
):
    model.eval()
    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        return model_forward_from_batch(
            model,
            b,
            return_debug=True,
            return_full_temporal_attention=full_attention,
            temporal_memory_ablation=memory_ablation,
            detection_graph_ablation=graph_ablation,
        )

def collect_snapshot(
    tag,
    *,
    b=device_batch,
    memory_ablation="full",
    graph_ablation="full",
    save_tables=True,
    compute_native=True,
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    started = time.perf_counter()
    output = run_full_forward(
        model,
        b,
        memory_ablation=memory_ablation,
        graph_ablation=graph_ablation,
        full_attention=True,
    )

    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        losses = eval_criterion(
            output,
            b["targets"],
        )

    probe = run_matching_probe(
        output,
        b["targets"],
    )
    match = probe.matches[0]

    valid = ~output.query_padding_mask[0]
    probs = output.exist_logits[0].sigmoid().detach().float().cpu()
    positive = torch.zeros_like(valid.detach().cpu())
    positive[match.pred_indices.detach().cpu()] = True
    valid_cpu = valid.detach().cpu()

    scores_np = probs[valid_cpu].numpy()
    labels_np = positive[valid_cpu].long().numpy()

    matched_scores = probs[positive]
    unmatched_scores = probs[valid_cpu & ~positive]

    dense_fg = (
        output.dense_outputs["foreground_logits"][0, 0]
        .sigmoid()
        .detach()
    )
    dense_boundary = (
        output.dense_outputs["boundary_logits"][0, 0]
        .sigmoid()
        .detach()
    )
    gt_fg = target["foreground"].to(device) > 0.5
    gt_boundary = target["boundary"].to(device) > 0.5
    foreground_dice_hard = hard_dice(dense_fg > 0.5, gt_fg)
    boundary_dice_hard = hard_dice(dense_boundary > 0.5, gt_boundary)
    del gt_fg, gt_boundary, dense_fg, dense_boundary

    matched_by_type = defaultdict(int)
    for q in match.pred_indices.detach().cpu().tolist():
        matched_by_type[
            QUERY_NAMES[int(output.query_types[0, q].detach().cpu())]
        ] += 1

    component_table = component_attention_table(output, SOURCE_ID)
    attention_top, frame_mass, diversity = query_attention_analysis(
        output,
        SOURCE_ID,
    )
    representation = source_representation_metrics(
        output,
        SOURCE_ID,
    )

    if compute_native:
        native_summary, native_query_table = source_native_metrics(
            model,
            output,
            probe,
            SOURCE_ID,
        )
    else:
        native_summary = {}
        native_query_table = pd.DataFrame()

    summary = {
        "tag": tag,
        "step": int(trainer.global_step),
        "memory_ablation": memory_ablation,
        "graph_ablation": graph_ablation,
        "loss": float(losses["loss"].detach().cpu()),
        "foreground_dice_hard": foreground_dice_hard,
        "boundary_dice_hard": boundary_dice_hard,
        "coarse_soft_dice": 1.0 - float(
            losses.get("dice_coarse", torch.tensor(float("nan")))
            .detach()
            .cpu()
        ),
        "native_soft_dice": 1.0 - float(
            losses.get("dice_hi", torch.tensor(float("nan")))
            .detach()
            .cpu()
        ),
        "exist_matched_mean": (
            float(matched_scores.mean())
            if matched_scores.numel()
            else float("nan")
        ),
        "exist_unmatched_mean": (
            float(unmatched_scores.mean())
            if unmatched_scores.numel()
            else float("nan")
        ),
        "exist_auc": binary_auc(
            scores_np,
            labels_np,
        ),
        "accepted_q_0p5": int(
            ((probs >= 0.5) & valid_cpu).sum()
        ),
        "matched_gt": int(match.target_indices.numel()),
        "all_gt_matched": bool(
            match.target_indices.numel()
            == len(target["ids"])
        ),
        "matched_primary": int(matched_by_type["primary"]),
        "matched_split": int(matched_by_type["split"]),
        "matched_temporal": int(matched_by_type["temporal"]),
        "matched_discovery": int(matched_by_type["discovery"]),
        "candidate_edges_active": int(
            output.debug["candidate_detection_edge_count"]
            .detach()
            .cpu()
        ),
        "accepted_edges_available": int(
            output.debug["accepted_detection_edge_count"]
            .detach()
            .cpu()
        ),
        "peak_cuda_gib": float(
            torch.cuda.max_memory_allocated()
            / 1024**3
        ),
        "elapsed_s": float(time.perf_counter() - started),
        **native_summary,
    }

    for layer, metrics in diversity.items():
        for key, value in metrics.items():
            summary[f"layer{layer}_{key}"] = value

    if not component_table.empty:
        for key, value in component_table.attrs.items():
            summary[f"component9_{key}"] = value

    if save_tables:
        if not component_table.empty:
            component_table.to_csv(
                RUN_DIR / f"{tag}_source9_component_attention.csv",
                index=False,
            )
        attention_top.to_csv(
            RUN_DIR / f"{tag}_source9_query_attention_top.csv",
            index=False,
        )
        frame_mass.to_csv(
            RUN_DIR / f"{tag}_source9_attention_by_frame.csv",
            index=False,
        )
        representation.to_csv(
            RUN_DIR / f"{tag}_source9_representation.csv",
            index=False,
        )
        if not native_query_table.empty:
            native_query_table.to_csv(
                RUN_DIR / f"{tag}_source9_native_assignments.csv",
                index=False,
            )

    return {
        "summary": summary,
        "component_table": component_table,
        "attention_top": attention_top,
        "frame_mass": frame_mass,
        "diversity": diversity,
        "representation": representation,
        "native_query_table": native_query_table,
        "node_tokens": (
            None
            if output.debug["node_memory"] is None
            else output.debug["node_memory"]["tokens"].float().cpu()
        ),
        "node_tracklet_id": (
            None
            if output.debug["node_memory"] is None
            else output.debug["node_memory"]["tracklet_id"].cpu()
        ),
        "best_current_component_id": (
            None
            if output.debug.get("best_current_component_id") is None
            else output.debug["best_current_component_id"].cpu()
        ),
    }

## 4. Snapshot the migrated step-30 model before new temporal training

This is the exact baseline against which the hierarchical-memory stages will be compared.

In [ ]:
snapshots = {}
stage_rows = []

snapshots["step30_migrated_spatial"] = collect_snapshot(
    "step30_migrated_spatial",
)
stage_rows.append(
    snapshots["step30_migrated_spatial"]["summary"]
)

display(
    pd.DataFrame(stage_rows).T
)

print()
print("Source-9 component top historical observations:")
display(
    snapshots["step30_migrated_spatial"]["component_table"].head(10)
)

print()
print("Source-9 query temporal attention:")
display(
    snapshots["step30_migrated_spatial"]["attention_top"]
)

## 5. Train from step 30 to step 85 with the existing curriculum

The optimizer is new, but the model starts from the migrated spatial checkpoint.

Assignment history is recorded after every query/native/joint optimizer step so we can measure Hungarian target churn.

In [ ]:
BOUNDARIES = {
    50: "step50_temporal_dense",
    70: "step70_query_bootstrap",
    80: "step80_native_bootstrap",
    85: "step85_joint",
}

CHECKPOINT_NAMES = {
    50: "checkpoint_temporal_dense.pt",
    70: "checkpoint_query_bootstrap.pt",
    80: "checkpoint_native_bootstrap.pt",
    85: "checkpoint_joint.pt",
}

assignment_history = []
training_log = []

def capture_assignment_history(step, stage):
    was_training = model.training
    model.eval()
    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        out = model_forward_from_batch(
            model,
            device_batch,
            return_debug=False,
        )
    probe = run_matching_probe(
        out,
        device_batch["targets"],
    )
    rows = source_assignment_rows(
        out,
        probe,
        step,
        stage,
        SOURCE_ID,
    )
    if was_training:
        model.train()
    del out
    return rows

started_training = time.perf_counter()

while trainer.global_step < 85:
    metrics = trainer.train_step(device_batch)
    after = trainer.global_step
    stage = trainer.curriculum_stage.name

    training_log.append({
        "step": after,
        "stage": stage,
        **metrics,
    })

    if (
        after % LOG_EVERY == 0
        or after in BOUNDARIES
        or after == 31
    ):
        compact = {
            key: value
            for key, value in metrics.items()
            if key in {
                "loss",
                "dice_coarse",
                "dice_hi",
                "exist",
                "center",
                "foreground",
                "boundary",
            }
        }
        print(
            f"step {after:3d} | {stage:17s} | "
            + " ".join(
                f"{k}={v:.5f}"
                for k, v in compact.items()
            )
        )

    if (
        stage in {
            "query_bootstrap",
            "native_bootstrap",
            "joint",
        }
        and after % ASSIGNMENT_EVERY == 0
    ):
        assignment_history.extend(
            capture_assignment_history(
                after,
                stage,
            )
        )

    if after in BOUNDARIES:
        tag = BOUNDARIES[after]
        snapshot = collect_snapshot(tag)
        snapshots[tag] = snapshot
        stage_rows.append(snapshot["summary"])

        save_checkpoint(
            RUN_DIR / CHECKPOINT_NAMES[after],
            model=model,
            optimizer=trainer.optimizer,
            scheduler=trainer.scheduler,
            scaler=trainer.scaler,
            step=trainer.global_step,
            epoch=0,
            config=cfg,
            extra={"notebook": 15, "snapshot": tag},
        )

        print()
        print("Snapshot:", tag)
        display(pd.DataFrame([snapshot["summary"]]))
        print()
        display(snapshot["attention_top"])
        print()

print(
    "Training elapsed:",
    round((time.perf_counter() - started_training) / 60, 2),
    "minutes",
)

## 6. Stage-to-stage summary

The important temporal columns are the attention similarities and JS divergence.

Lower sibling attention cosine and higher JS divergence mean different split hypotheses are using more distinct temporal evidence.

In [ ]:
stage_df = pd.DataFrame(stage_rows)
stage_df.to_csv(
    RUN_DIR / "stage_metrics.csv",
    index=False,
)

display(stage_df)

plot_cols = [
    "foreground_dice_hard",
    "boundary_dice_hard",
    "native_soft_dice",
    "exist_auc",
    "native_pair_dice_mean",
    "assigned_mean_dice",
]

available = [c for c in plot_cols if c in stage_df.columns]
if available:
    ax = stage_df.set_index("tag")[available].plot(
        marker="o",
        figsize=(11, 5),
    )
    ax.set_title("Notebook 15 stage metrics")
    ax.grid(True, alpha=0.25)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Source-9 temporal specialization across stages

This table follows the path:

`temporal attention → query embedding → center separation → mask separation`.

In [ ]:
representation_frames = []

for tag, snapshot in snapshots.items():
    rep = snapshot["representation"].copy()
    rep.insert(0, "tag", tag)

    diversity = snapshot["diversity"]
    for layer in range(3):
        if layer in diversity:
            for key, value in diversity[layer].items():
                rep.loc[
                    rep["layer"] == layer,
                    key,
                ] = value

    representation_frames.append(rep)

representation_df = pd.concat(
    representation_frames,
    ignore_index=True,
)
representation_df.to_csv(
    RUN_DIR / "source9_layer_specialization.csv",
    index=False,
)

display(representation_df)

In [ ]:
attention_frames = []

for tag, snapshot in snapshots.items():
    frame = snapshot["frame_mass"].copy()
    frame.insert(0, "tag", tag)
    attention_frames.append(frame)

attention_by_frame_df = pd.concat(
    attention_frames,
    ignore_index=True,
)
attention_by_frame_df.to_csv(
    RUN_DIR / "source9_attention_by_frame_all_stages.csv",
    index=False,
)

display(attention_by_frame_df)

## 8. Hungarian assignment churn

A useful temporal representation can still fail to specialize masks if the same query is assigned to different GT cells from step to step.

This section measures that explicitly.

In [ ]:
assignment_df = pd.DataFrame(assignment_history)
assignment_df.to_csv(
    RUN_DIR / "source9_assignment_history.csv",
    index=False,
)

display(assignment_df.head(40))

if not assignment_df.empty:
    churn_rows = []
    for query_label, group in assignment_df.groupby("query_label"):
        ordered = group.sort_values("step")
        gt = ordered["gt_id"].to_numpy()
        switches = int(
            np.count_nonzero(gt[1:] != gt[:-1])
        ) if len(gt) > 1 else 0
        churn_rows.append({
            "query_label": query_label,
            "observations": len(group),
            "unique_targets": int(
                group["gt_id"].nunique()
            ),
            "switches": switches,
            "final_gt_id": int(
                ordered.iloc[-1]["gt_id"]
            ),
        })

    churn_df = pd.DataFrame(churn_rows).sort_values(
        "query_label"
    )
    churn_df.to_csv(
        RUN_DIR / "source9_assignment_churn.csv",
        index=False,
    )
    display(churn_df)

    pivot = assignment_df.pivot_table(
        index="query_label",
        columns="step",
        values="gt_id",
        aggfunc="last",
    )
    plt.figure(figsize=(14, 5))
    image = plt.imshow(
        pivot.to_numpy(),
        aspect="auto",
        interpolation="nearest",
    )
    plt.colorbar(image, label="GT ID (-1 = unmatched)")
    plt.yticks(
        np.arange(len(pivot.index)),
        pivot.index,
    )
    tick_step = max(1, len(pivot.columns) // 12)
    plt.xticks(
        np.arange(len(pivot.columns))[::tick_step],
        pivot.columns[::tick_step],
        rotation=45,
    )
    plt.xlabel("training step")
    plt.ylabel("source-9 seeded query")
    plt.title("Source-9 Hungarian assignment history")
    plt.tight_layout()
    plt.show()

## 9. Build a true legacy-like graph ablation

`accepted_only` is intentionally extreme: it keeps only Trackastra-accepted directed relations.

The old detection graph also had bounded same-frame neighbours. For a fairer comparison, this notebook constructs:

`accepted Trackastra relations + up to 6 nearest same-frame neighbours within 2.5 dref`

from the already cached complete graph, without rebuilding preprocessing.

In [ ]:
def make_legacy_like_graph_batch(cpu_batch):
    result = dict(cpu_batch)

    edge_index = cpu_batch["graph_edge_index"].detach().cpu()
    edge_attr = cpu_batch["graph_edge_attr"].detach().cpu()
    observed = cpu_batch["node_observed_ref_um"].detach().cpu().float()
    times = cpu_batch["node_time_offset"].detach().cpu().round().long()
    dref = float(cpu_batch["dref_um"][0])

    keep = edge_attr[:, 14] > 0.5

    pair_to_edge = {
        (int(source), int(destination)): edge
        for edge, (source, destination) in enumerate(
            edge_index.T.tolist()
        )
    }

    radius_um = (
        float(cfg.temporal.spatial_neighbor_radius_dref)
        * dref
    )
    k = int(cfg.temporal.k_spatial_neighbors)
    indices = torch.arange(len(observed))

    for source in range(len(observed)):
        candidates = torch.nonzero(
            (times == times[source])
            & (indices != source),
            as_tuple=False,
        ).flatten()

        if candidates.numel() == 0:
            continue

        distances = torch.linalg.vector_norm(
            observed[candidates] - observed[source],
            dim=-1,
        )
        valid = candidates[distances <= radius_um]
        if valid.numel() == 0:
            continue

        valid_distances = torch.linalg.vector_norm(
            observed[valid] - observed[source],
            dim=-1,
        )
        order = valid[
            torch.argsort(valid_distances)[:k]
        ]

        for destination in order.tolist():
            edge = pair_to_edge.get(
                (int(source), int(destination))
            )
            if edge is not None:
                keep[edge] = True

    selected = torch.nonzero(
        keep,
        as_tuple=False,
    ).flatten()

    result["graph_edge_index"] = cpu_batch[
        "graph_edge_index"
    ][:, selected]
    result["graph_edge_attr"] = cpu_batch[
        "graph_edge_attr"
    ][selected]

    return result

legacy_cpu_batch = make_legacy_like_graph_batch(batch)
legacy_device_batch = dict(device_batch)
legacy_device_batch["graph_edge_index"] = legacy_cpu_batch[
    "graph_edge_index"
].to(device=device, non_blocking=True)
legacy_device_batch["graph_edge_attr"] = legacy_cpu_batch[
    "graph_edge_attr"
].to(device=device, non_blocking=True)

print(
    "Full candidate edges :",
    batch["graph_edge_index"].shape[1],
)
print(
    "Accepted only        :",
    int((batch["graph_edge_attr"][:, 14] > 0.5).sum()),
)
print(
    "Legacy-like edges    :",
    legacy_cpu_batch["graph_edge_index"].shape[1],
)

## 10. Same-weight causal ablations at the final checkpoint

These are **not independently trained models**. Every trial uses the exact same step-85 weights.

Interpretation:

- `full vs tracklet_only` → value of retaining fine observations;
- `full vs shuffle_node` → value of correct node identity;
- `node_only vs tracklet_only` → fine observation vs trajectory abstraction;
- `full vs legacy_like_graph` → value of the complete candidate topology;
- `accepted_only` → intentionally extreme Trackastra-only graph.

In [ ]:
ABLATIONS = {
    "full": {
        "batch": device_batch,
        "memory": "full",
        "graph": "full",
    },
    "zero_node": {
        "batch": device_batch,
        "memory": "zero_node",
        "graph": "full",
    },
    "shuffle_node": {
        "batch": device_batch,
        "memory": "shuffle_node",
        "graph": "full",
    },
    "tracklet_only": {
        "batch": device_batch,
        "memory": "tracklet_only",
        "graph": "full",
    },
    "node_only": {
        "batch": device_batch,
        "memory": "node_only",
        "graph": "full",
    },
    "accepted_only": {
        "batch": device_batch,
        "memory": "full",
        "graph": "accepted_only",
    },
    "legacy_like_graph": {
        "batch": legacy_device_batch,
        "memory": "full",
        "graph": "full",
    },
}

ablation_results = {}
ablation_rows = []
variant_node_tokens = {}

for name, settings in ABLATIONS.items():
    print("\n---", name, "---")

    result = collect_snapshot(
        f"ablation_{name}",
        b=settings["batch"],
        memory_ablation=settings["memory"],
        graph_ablation=settings["graph"],
        save_tables=False,
        compute_native=True,
    )

    ablation_results[name] = result
    row = dict(result["summary"])
    row["variant"] = name
    ablation_rows.append(row)

    if result["node_tokens"] is not None:
        variant_node_tokens[name] = result["node_tokens"]

ablation_df = pd.DataFrame(ablation_rows)

full_loss = float(
    ablation_df.loc[
        ablation_df["variant"] == "full",
        "loss",
    ].iloc[0]
)
ablation_df["loss_delta_vs_full"] = (
    ablation_df["loss"] - full_loss
)

ablation_df.to_csv(
    RUN_DIR / "final_same_weight_ablations.csv",
    index=False,
)

columns = [
    "variant",
    "loss",
    "loss_delta_vs_full",
    "foreground_dice_hard",
    "native_soft_dice",
    "exist_auc",
    "native_pair_dice_mean",
    "assigned_mean_dice",
    "matched_gt",
    "candidate_edges_active",
]
columns += [
    c for c in [
        "layer0_attention_cos_mean",
        "layer1_attention_cos_mean",
        "layer2_attention_cos_mean",
        "layer2_attention_js_mean",
        "layer2_unique_top_nodes",
        "layer2_unique_top_tracklets",
    ]
    if c in ablation_df.columns
]
display(ablation_df[columns])

## 11. How much does graph topology alter the learned node memory?

This operates *before* query attention.

If the full candidate graph and the legacy/accepted graphs produce almost identical node tokens, the extra candidate topology is not contributing much yet.

In [ ]:
node_representation_rows = []

full_tokens = variant_node_tokens.get("full")

if full_tokens is not None:
    full_result = ablation_results["full"]
    node_tracklet = full_result["node_tracklet_id"]
    best_component = full_result["best_current_component_id"]

    source9_node_mask = None
    if (
        node_tracklet is not None
        and best_component is not None
        and len(best_component)
    ):
        safe_tracklet = node_tracklet.clamp(
            0,
            len(best_component) - 1,
        )
        source9_node_mask = (
            best_component[safe_tracklet] == SOURCE_ID
        )

    for name in (
        "accepted_only",
        "legacy_like_graph",
    ):
        other = variant_node_tokens.get(name)
        if other is None:
            continue

        cosine = F.cosine_similarity(
            full_tokens.float(),
            other.float(),
            dim=-1,
        )
        l2 = torch.linalg.vector_norm(
            full_tokens.float() - other.float(),
            dim=-1,
        )

        row = {
            "variant": name,
            "all_node_cosine_mean": float(cosine.mean()),
            "all_node_cosine_median": float(cosine.median()),
            "all_node_l2_mean": float(l2.mean()),
        }

        if (
            source9_node_mask is not None
            and source9_node_mask.any()
        ):
            row.update({
                "source9_related_nodes": int(
                    source9_node_mask.sum()
                ),
                "source9_node_cosine_mean": float(
                    cosine[source9_node_mask].mean()
                ),
                "source9_node_l2_mean": float(
                    l2[source9_node_mask].mean()
                ),
            })

        node_representation_rows.append(row)

node_representation_df = pd.DataFrame(
    node_representation_rows
)
node_representation_df.to_csv(
    RUN_DIR / "node_memory_graph_ablation.csv",
    index=False,
)
display(node_representation_df)

## 12. Final source-9 attention inspection

The table below should be read biologically.

Look for whether different split slots eventually attend different observations from T-2 / T-1 / T+1 / T+2, especially cells that visibly separate from the large cluster in another frame.

In [ ]:
final_snapshot = snapshots["step85_joint"]

print("Component 9 temporal attention:")
display(
    final_snapshot["component_table"].head(15)
)

print("\nPer-query top temporal evidence:")
display(
    final_snapshot["attention_top"].sort_values(
        ["layer", "query_label"]
    )
)

print("\nAttention mass by frame:")
display(
    final_snapshot["frame_mass"].sort_values(
        ["layer", "query_label"]
    )
)

print("\nRepresentation path:")
display(
    final_snapshot["representation"]
)

print("\nFinal source-9 native assignments:")
display(
    final_snapshot["native_query_table"]
)

## 13. Save compact JSON summaries

The detailed row-level diagnostics are already saved as CSV files. This cell writes compact JSON summaries for later comparison.

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    if isinstance(value, float) and (
        math.isnan(value) or math.isinf(value)
    ):
        return None
    return value

stage_json = {
    tag: json_safe(snapshot["summary"])
    for tag, snapshot in snapshots.items()
}
with (
    RUN_DIR / "stage_metrics.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        stage_json,
        handle,
        indent=2,
    )

ablation_json = {
    name: json_safe(result["summary"])
    for name, result in ablation_results.items()
}
with (
    RUN_DIR / "final_ablations.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        ablation_json,
        handle,
        indent=2,
    )

pd.DataFrame(training_log).to_csv(
    RUN_DIR / "training_log.csv",
    index=False,
)

print("Saved diagnostics to:")
print(RUN_DIR)
for path in sorted(RUN_DIR.glob("*")):
    print(" ", path.name)

## 14. Decision guide

Use the outputs above to decide the next intervention.

| Observation | Interpretation |
|---|---|
| Full > shuffle, split attention differs, centers/masks separate | Hierarchical temporal memory is working |
| Attention differs, but embeddings become similar later | Decoder homogenization remains |
| Attention differs, but Hungarian assignments keep switching | Matching / set-supervision identity instability remains |
| Attention remains nearly identical and shuffle has no effect | Temporal identity is not being learned meaningfully |
| Full > legacy-like graph | Candidate relations beyond the old graph are useful |
| Node-only > tracklet-only | Fine observations are especially valuable |
| Tracklet-only > node-only | Trajectory abstraction carries more of the useful signal |

**Do not change the architecture merely because total loss still falls slowly.**

The main evidence is the causal path:

`correct temporal identity → distinct attention → stable query identity → stable assignment → distinct masks`.

## 15. Optional Napari: show the top attended historical nodes for one final split query

This cell is disabled by default. Set `OPEN_NAPARI = True` after the numerical analysis.

It overlays the actual Trackastra trajectories with the top historical detections attended by one selected source-9 query.

In [ ]:
OPEN_NAPARI = True
SELECT_LAYER = 2
SELECT_QUERY_LABEL = "split0"
TOP_NODES = 20

if OPEN_NAPARI:
    import pickle
    import napari

    from trackastra.tracking import graph_to_napari_tracks
    from learned.stirnet.debugging.acceptance.first_overfit import (
        _roi_with_all_cells,
    )

    raw_movie = np.load(
        DATA_DIR / "raw_movie.npy",
        mmap_mode="r",
    )
    instance_movie = np.load(
        DATA_DIR / "instance_movie.npy",
        mmap_mode="r",
    )
    gt_movie = np.load(
        DATA_DIR / "gt_movie.npy",
        mmap_mode="r",
    )

    spacing = (
        batch["spacing_um"][0]
        .detach()
        .cpu()
        .numpy()
    )
    roi, roi_low, roi_high = _roi_with_all_cells(
        instance_movie,
        gt_movie,
        spacing,
    )

    raw_roi = raw_movie[
        (slice(None),) + roi
    ]
    instance_roi = instance_movie[
        (slice(None),) + roi
    ]

    with (
        DATA_DIR / "trackastra" / "track_graph.pkl"
    ).open("rb") as handle:
        track_graph = pickle.load(handle)

    track_array, lineage_graph, _ = graph_to_napari_tracks(
        track_graph
    )
    track_array = np.asarray(
        track_array,
        dtype=np.float32,
    ).copy()
    track_array[:, 2:5] -= roi_low[None].astype(
        np.float32
    )

    final_output = run_full_forward(
        model,
        device_batch,
        full_attention=True,
    )

    layer_debug = final_output.debug[
        "query_temporal_attention"
    ][SELECT_LAYER]
    slots = (
        layer_debug["query_slot_index"]
        .detach()
        .cpu()
        .long()
    )
    sources = (
        layer_debug["source_instance_id"]
        .detach()
        .cpu()
        .long()
    )
    qtypes = (
        layer_debug["query_type"]
        .detach()
        .cpu()
        .long()
    )

    _, labels = source_query_labels(
        final_output,
        SOURCE_ID,
    )
    wanted_query = [
        q for q, label in labels.items()
        if label == SELECT_QUERY_LABEL
    ][0]

    row = torch.nonzero(
        (slots == wanted_query)
        & (sources == SOURCE_ID)
        & (
            (qtypes == QUERY_PRIMARY)
            | (qtypes == QUERY_SPLIT)
        ),
        as_tuple=False,
    ).flatten()
    if len(row) != 1:
        raise RuntimeError(
            "Could not uniquely locate selected query debug row."
        )
    row = int(row[0])

    weights = (
        layer_debug["node"]["full_weights"][row]
        .float()
        .detach()
        .cpu()
    )
    node_memory = final_output.debug["node_memory"]
    node_time = (
        node_memory["time_offset"]
        .detach()
        .cpu()
        .numpy()
    )
    node_pos_um = (
        node_memory["observed_ref_um"]
        .detach()
        .cpu()
        .numpy()
    )

    shape = np.asarray(
        instance_roi.shape[-3:],
        dtype=np.float32,
    )
    center_um = (
        0.5 * (shape - 1)
        * spacing
    )
    node_vox = (
        node_pos_um + center_um[None]
    ) / spacing[None]

    order = torch.argsort(
        weights,
        descending=True,
    )[:TOP_NODES].numpy()

    points = np.concatenate(
        [
            (node_time[order] + 2)[:, None],
            node_vox[order],
        ],
        axis=1,
    )
    point_weights = weights[order].numpy()

    viewer = napari.Viewer(
        ndisplay=3,
        title=(
            f"Notebook 15 | {SELECT_QUERY_LABEL} "
            f"layer {SELECT_LAYER}"
        ),
    )
    scale_tzyx = (1.0, *spacing.tolist())

    viewer.add_image(
        raw_roi,
        name="Raw movie",
        scale=scale_tzyx,
        rendering="mip",
    )
    viewer.add_labels(
        instance_roi,
        name="CC instances",
        scale=scale_tzyx,
        visible=False,
    )
    viewer.add_tracks(
        track_array,
        graph=lineage_graph,
        name="Trackastra tracks",
        scale=scale_tzyx,
        tail_length=20,
    )
    viewer.add_points(
        points,
        name="Top attended historical nodes",
        scale=scale_tzyx,
        size=5,
        features={"weight": point_weights},
        text={
            "string": "{weight:.3f}",
            "size": 8,
        },
    )

    viewer.dims.set_current_step(
        0,
        int(raw_roi.shape[0]) - 1,
    )
    napari.run()